## Testing edits to reassortment MLE code

In [2]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize_scalar
import pandas as pd
import ast
import tskit

In [103]:
class SCAR(object):
    
    """
        Model class for the structured coalescent with ancestral recombination (SCAR)
        Ancestral states can be given (known) or marginalized over (unknown).
        This version relaxes rules about the ARG:
        Specifically coal nodes need not be strictly bifurcating and can have any number of children.
        But all children must have one and only one parent.
    """
    
    def __init__(self,reassortment_rate,M,Ne,genome_length,**kwargs):
        
        '''             
            Parameters: 
                reassortment_rate (float): reassortment rate per lineage per site
                M (2D list/array): migration rates (forward-time) between subpopulations 
                Ne (1D list/array): effective population size of each subpopulation
                genome_length (int): The number of segments K, with unit segments [0,1), ..., [K-1, K).
               
            Optional keyword arguments:
                bounds (tuple): lower and upper bounds on estimated parameter given as (lower,upper)
                dt_step (str): integration time step used to compute lineage state probabilities
                known_ancestral_states (boolean): ancestral states need to be given in tree series (ts) if True
        '''
        
        # Model params
        self.reassortment_rate = reassortment_rate
        self.M = np.transpose(np.array(M)) # transpose to get reverse time matrix        
        self.Ne = np.array(Ne)
        self.genome_length = genome_length
        
        # Likelihood calculation params
        self.bounds = kwargs.get('bounds', (0, np.inf))
        self.dt_step = kwargs.get('dt_step', 0.1)
        self.known_ancestral_states = kwargs.get('known_ancestral_states', False)
        
    def _get_line_segment_count(self,line,children,segments):
        
        """
            Compute the number of segments ancestral to the sample and thus eligible to undergo reassortment
        """
        
        line_segments = segments[children == line]
        if len(line_segments) == 0:
            return 0
        all_segs = set()
        for seg_list in line_segments:
            all_segs.update(seg_list)
        return len(all_segs)  

In [107]:
# Read in the tskit object!

samples = 10
genome_length = 3
reassortment_rate = 0.01
Ne = 10  # effective pop sizes
#M = [[0.0,0.25],[0.25,0.0]]  # migration rate matrix
M = [[0]]

ts = tskit.load("arg_w_tables.trees")

# Initialize SCAR model class
bounds = (0.0,0.1)
scar_model = SCAR(reassortment_rate,M,Ne,genome_length,bounds=bounds)

# Check numerical optimization for MLE of single param
#mle = scar_model.opt_MLE(ts)
#print(mle)

In [108]:
#    def compute_neg_log_like(self,reassortment_rate,table_nodes,table_edges):
self = scar_model
reassortment_rate = 0.01
table_nodes = "arg_w_tables.trees.nodes.csv"
table_edges = "arg_w_tables.trees.edges.csv"

In [114]:
pops = 1
#samples = ts.num_samples
#states = [st.id for st in ts.populations()]

# Get transition rate matrix
Q = self.M - np.diag(np.sum(self.M,axis=1)) # set diagonals to negative row sums

# Read in tables of nodes and edges for ARG
nodes_df = pd.read_csv(table_nodes)
edges_df = pd.read_csv(table_edges)

In [115]:
children = edges_df["child"].to_numpy()
parents = edges_df["parent"].to_numpy()
segments = edges_df["segments"].apply(ast.literal_eval).to_numpy()

In [116]:
# Init lineage arrays
active_lines = [] # active lines in ARG
active_segments = [] # tracks ancestral segments that can reassort on active_lines
line_state_probs = [] # lineage state probabilities for active lines
log_like = 0.0 # log likelihood of full tree

In [117]:
trace = []

# Iterate through each event in ARG nodes working backwards through time
for i in np.arange(0,len(nodes_df)):

    # Get time of event and time of next event"
    event_time = nodes_df.time[i]
    if (i+1 < len(nodes_df)): # if not at final event
        next_time = nodes_df.time[i+1]
    else:
        next_time = nodes_df.time[i]
    t_elapsed = next_time - event_time # time elapsed between events

    # Determine event type from tskit event.flags
    event_type = None
    if nodes_df.type[i] == 'sample':
        event_type = 'sample'
    if nodes_df.type[i] == 'coalescence':
        event_type = 'coalescent'
    if nodes_df.type[i] == 'recombination':
        event_type = 'recombination'
    #if event.flags == 262144:
    #    event_type = 'hidden_coalescent'
    #if event.flags == 524288:
    #    event_type = 'migration'

    # Initialize prob of observing events or no events
    event_prob = 1.0
    prob_no_coal = 1.0
    prob_no_mig = 1.0
    prob_no_reassort = 1.0

    # Update active lineages based on event type: coalescent/sampling/migration events
    if 'sample' == event_type:

        # Add sampled lineage
        active_lines.append(i)
        active_segments.append(self._get_line_segment_count(i,children,segments))
        state_probs = np.zeros(pops)
        #if event.population == -1: # we never assigned populations
        state_probs[0] = 1.0 # set prob to 1.0 for sampled state
        #else:
        #    state_probs[event.population] = 1.0 # set prob to 1.0 for sampled state
        line_state_probs.append(state_probs)            

    if 'coalescent' == event_type:

        # Get children of parent node at coalescent event
        coal_children = children[parents == i] # parent has id == i in parent column of edges table

        # Get uniique children b/c the same parent/child edge may occur more than once in the tree series if not in contiguous local trees
        coal_children = np.unique(coal_children)


        # Find coal_children in active_lines
        coal_children = [x for x in coal_children if x in active_lines]
        child_indexes = [active_lines.index(x) for x in coal_children]

        # Compute coalescent event prob for arbitrary number of children 
        coal_probs = np.ones(pops)
        for child_idx in child_indexes:
            coal_probs *= line_state_probs[child_idx]
        coal_probs = coal_probs / self.Ne
        lambda_sum = sum(coal_probs)
        event_prob = lambda_sum

        # Compute new parent state probs
        if self.known_ancestral_states:
            parent_probs = np.zeros(pops)
            parent_probs[event.population] = 1.0
        else:
            parent_probs = coal_probs / lambda_sum # renormalize probs

        # Update lineage arrays - overwriting child1 with parent
        active_lines[child_indexes[0]] = i # name of parent
        active_segments[child_indexes[0]] = self._get_line_segment_count(i,children,segments)
        line_state_probs[child_indexes[0]] = parent_probs
        child_indexes.pop(0) # remove first index given to parent
        for child_idx in sorted(child_indexes, reverse=True): # remove in reverse order so indexes don't change
            del active_lines[child_idx]
            del active_segments[child_idx]
            del line_state_probs[child_idx]

    #if 'hidden_coalescent' == event_type:

        # Hidden coalescent in ARG not observed in local trees - only need to update active_lines but nothing else

    #    coal_children = children[parents == i]
    #    coal_children = np.unique(coal_children)
    #    child1 = coal_children[0]
    #    child2 = coal_children[1]
    #    child1_idx = active_lines.index(child1)
    #    child2_idx = active_lines.index(child2)

        # Compute likelihood of coalescent event
    #    p1 = line_state_probs[child1_idx]
    #    p2 = line_state_probs[child2_idx]
    #    coal_probs = (p1 * p2) / self.Ne
    #    lambda_sum = sum(coal_probs)
    #    event_prob = lambda_sum

        # Compute new parent state probs"
    #    if self.known_ancestral_states:
    #        parent_probs = np.zeros(pops)
    #        parent_probs[event.population] = 1.0
    #    else:
    #        parent_probs = coal_probs / lambda_sum

        # Update lineage arrays - overwriting child1 with parent"
    #    active_lines[child1_idx] = i # name of parent
    #    active_segments[child1_idx] = self._get_line_segment_count(i,children,segments)
    #    line_state_probs[child1_idx] = parent_probs
    #    del active_lines[child2_idx]
    #    del active_segments[child2_idx]
    #    del line_state_probs[child2_idx]

    if 'recombination' == event_type:

        # Find which currently-active lineage (by its OLD id) is passing through this event
        child = children[parents == i]
        child = np.unique(child)
        assert len(child) == 1
        child = child[0]

        if child in active_lines:

            # Node i's OWN parents are what this event splits into
            recomb_parents = parents[children == i]
            recomb_parents = np.unique(recomb_parents)

            child_idx = active_lines.index(child)

            if len(recomb_parents) == 1:
                # genuinely unobservable - all segments went to the same parent
                parent = recomb_parents[0]
                active_lines[child_idx] = parent
                active_segments[child_idx] = self._get_line_segment_count(parent, children, segments)
                event_prob = 1.0

            else:
                assert len(recomb_parents) == 2, \
                    f"Node {i}: expected 1 or 2 parents, got {len(recomb_parents)}"

                left_parent, right_parent = recomb_parents[0], recomb_parents[1]

                links = active_segments[child_idx]  # segment count of the pre-split lineage
                event_prob = reassortment_rate * (1 - (0.5)**(links - 1))

                parent_probs = line_state_probs[child_idx]

                active_lines[child_idx] = left_parent
                active_segments[child_idx] = self._get_line_segment_count(left_parent, children, segments)
                line_state_probs[child_idx] = parent_probs

                active_lines.append(right_parent)
                active_segments.append(self._get_line_segment_count(right_parent, children, segments))
                line_state_probs.append(parent_probs)


    #if 'migration' == event_type:

        # Find migrating (child) lineage
    #    mig_child = children[parents == i] # parent has id == idx in parent column of edges table
    #    mig_child = np.unique(mig_child)

        # Get migration info from nodes list
    #    curr_state = populations[mig_child[0]]
    #    new_state = populations[i]

    #    migrant_idx = active_lines.index(mig_child) #change this for ts index

        # Update lineage arrays
    #    active_lines[migrant_idx] = idx # name of parent

        # Compute event prob
    #    if self.known_ancestral_states:
    #        new_probs = np.zeros(pops)
    #        new_probs[new_state] = 1.0 # event. population
    #        line_state_probs[migrant_idx] = new_probs
    #        event_prob = self.M[curr_state][new_state]
    #    else:
    #        event_prob = 1.0 # pretend as if we don't see migration events
                        
            
    trace.append({
        'i': i,
        'time': event_time,
        'type': event_type,
        't_elapsed': t_elapsed,
        'event_prob': event_prob,
        'active_lines': list(active_lines),
        'active_segments': list(active_segments),
        'n_active': len(active_lines),
        'log_like_running': log_like,
    })

    log_like += np.log(event_prob)


In [118]:
log_like

-42.30957789124816

In [102]:
edges_df

,parent,child,segments
0,10,3,"[0, 1, 2]"
1,11,10,"[1, 2]"
2,11,1,"[0, 1, 2]"
3,12,5,"[0, 1, 2]"
4,12,4,"[0, 1, 2]"
5,13,8,"[0, 1, 2]"
6,13,12,"[0, 1, 2]"
7,14,10,[0]
8,14,7,"[0, 1, 2]"
9,15,6,"[0, 1, 2]"


In [98]:
log_like

-27.63102111592855

In [120]:
trace_df = pd.DataFrame(trace)

In [121]:
trace_df

,i,time,type,t_elapsed,event_prob,active_lines,active_segments,n_active,log_like_running
0,0,0.000000,sample,0.000000,1.0000,[0],[3],1,0.000000
1,1,0.000000,sample,0.000000,1.0000,"[0, 1]","[3, 3]",2,0.000000
2,2,0.000000,sample,0.000000,1.0000,"[0, 1, 2]","[3, 3, 3]",3,0.000000
3,3,0.000000,sample,0.000000,1.0000,"[0, 1, 2, 3]","[3, 3, 3, 3]",4,0.000000
4,4,0.000000,sample,0.000000,1.0000,"[0, 1, 2, 3, 4]","[3, 3, 3, 3, 3]",5,0.000000
5,5,0.000000,sample,0.000000,1.0000,"[0, 1, 2, 3, 4, 5]","[3, 3, 3, 3, 3, 3]",6,0.000000
6,6,0.000000,sample,0.000000,1.0000,"[0, 1, 2, 3, 4, 5, 6]","[3, 3, 3, 3, 3, 3, 3]",7,0.000000
7,7,0.000000,sample,0.000000,1.0000,"[0, 1, 2, 3, 4, 5, 6, 7]","[3, 3, 3, 3, 3, 3, 3, 3]",8,0.000000
8,8,0.000000,sample,0.000000,1.0000,"[0, 1, 2, 3, 4, 5, 6, 7, 8]","[3, 3, 3, 3, 3, 3, 3, 3, 3]",9,0.000000
9,9,0.000000,sample,0.491937,1.0000,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3]",10,0.000000


In [54]:
event_prob = 1.0
prob_no_coal = 1.0
prob_no_mig = 1.0
prob_no_reassort = 1.0

In [95]:
trace_df = pd.DataFrame(trace)

In [8]:
children

array([ 3, 10,  1,  5,  4,  8, 12, 10,  7,  6, 11, 13,  9,  2,  0, 17, 17,
       18, 14, 19, 16, 20, 15, 21, 22, 23, 21])

In [9]:
parents

array([10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 18, 18, 19,
       19, 20, 20, 21, 22, 22, 23, 23, 24, 24])

In [10]:
segments

array([list([0, 1, 2]), list([1, 2]), list([0, 1, 2]), list([0, 1, 2]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([0]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([1, 2]),
       list([0]), list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]),
       list([0, 1, 2]), list([0, 1, 2]), list([0, 1, 2]), list([0, 1]),
       list([0, 1, 2]), list([2]), list([2])], dtype=object)

In [57]:
for i in np.arange(0,len(nodes_df)):
    event_time = nodes_df.time[i]
    if (i+1 < len(nodes_df)): # if not at final event
        next_time = nodes_df.time[i+1]
    else:
        next_time = nodes_df.time[i]
    t_elapsed = next_time - event_time # time elapsed between events
    
    # Determine event type from tskit event.flags
    event_type = None
    if nodes_df.type[i] == 'sample':
        event_type = 'sample'
    if nodes_df.type[i] == 'coalescence':
        event_type = 'coalescent'
    if nodes_df.type[i] == 'recombination':
        event_type = 'recombination'
                
      # Update active lineages based on event type: coalescent/sampling/migration events
    if 'sample' == event_type:

        # Add sampled lineage
        active_lines.append(i)
        active_segments.append(self._get_line_segment_count(i,children,segments))
        state_probs = np.zeros(pops)
        #if event.population == -1: # we never assigned populations
        state_probs[0] = 1.0 # set prob to 1.0 for sampled state
        #else:
        #    state_probs[event.population] = 1.0 # set prob to 1.0 for sampled state
        line_state_probs.append(state_probs)   

In [58]:
active_lines

[0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [59]:
active_segments

[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

In [61]:
i=10
event_time = nodes_df.time[i]
if (i+1 < len(nodes_df)): # if not at final event
    next_time = nodes_df.time[i+1]
else:
    next_time = nodes_df.time[i]
t_elapsed = next_time - event_time # time elapsed between events

In [62]:
event_type = None
if nodes_df.type[i] == 'sample':
    event_type = 'sample'
if nodes_df.type[i] == 'coalescence':
    event_type = 'coalescent'
if nodes_df.type[i] == 'recombination':
    event_type = 'recombination'

In [63]:
event_type

'recombination'

In [64]:
event_prob = 1.0
prob_no_coal = 1.0
prob_no_mig = 1.0
prob_no_reassort = 1.0

In [65]:
#if 'recombination' == event_type:
# Find child of parent node
child = children[parents == i]
child = np.unique(child)
assert len(child) == 1

In [66]:
child

array([3])

In [67]:
child = np.array(10)
recomb_parents = parents[children == child]


In [68]:
recomb_parents

array([11, 14])

In [69]:
 recomb_parents = np.unique(recomb_parents)

In [70]:
left_parent = recomb_parents[0]
right_parent = recomb_parents[1]

In [71]:
child_idx = active_lines.index(child)

ValueError: array(10) is not in list